<a href="https://colab.research.google.com/github/prinshu756/Exoplanet-detection/blob/main/notebooks/02_Light_Curve_Acquisition_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## This is the second notebook

In [1]:
!pip -q install lightkurve astropy astroquery pyarrow tqdm wotan

In [2]:
import os
import gc
import time
import logging
import warnings

from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.notebook import tqdm

import matplotlib.pyplot as plt

import lightkurve as lk

warnings.filterwarnings("ignore")

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [3]:
PROJECT_NAME = "Exoplanet-detection"

ROOT_DIR = Path("D:/MachineLearning/Exoplanet-detection")

CATALOG_DIR = ROOT_DIR / "data" / "processed"

LIGHTCURVE_DIR = ROOT_DIR / "data" / "raw" / "lightcurves"

LOG_DIR = ROOT_DIR / "logs"

MAX_TARGETS = 100      # None → Entire Catalog

MISSION = "TESS"

AUTHOR = "SPOC"

DOWNLOAD_TIMEOUT = 120

print("Configuration Loaded")

Configuration Loaded


In [4]:
LOG_FILE = LOG_DIR / "notebook_02.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler('D:/MachineLearning/Exoplanet-detection/logs/notebook_02.log'),
        logging.StreamHandler()
    ],
    force=True
)

logger = logging.getLogger("Notebook02")

logger.info("Notebook 02 Started")

2026-07-23 10:56:29,465 | INFO | Notebook 02 Started


In [5]:
catalog = pd.read_parquet(
    ROOT_DIR/"data/processed/master_catalog.parquet"
)

if MAX_TARGETS is not None:
    catalog = catalog.head(MAX_TARGETS)

print(f"Targets Loaded : {len(catalog):,}")

catalog.head()

Targets Loaded : 100


,tic_id,planet_name,host_star,ra,dec,orbital_period_days,planet_radius_earth,planet_mass_earth,stellar_teff,stellar_radius,stellar_mass,distance_pc,discovery_year,class_label,class_id,source_catalog
0,158722002,Kepler-1597 b,Kepler-1597,288.163969,42.481012,2.946542,1.06,1.20,6377.0,1.390,1.250,1221.050,2016,Planet,0,NASA Exoplanet Archive
1,121988890,Kepler-687 b,Kepler-687,289.315458,39.153291,20.505870,3.52,12.20,4841.0,0.730,0.770,633.660,2016,Planet,0,NASA Exoplanet Archive
2,350813882,Kepler-1596 b,Kepler-1596,285.557507,49.330087,66.373379,1.90,4.27,5706.0,0.920,0.950,2146.230,2016,Planet,0,NASA Exoplanet Archive
3,138728891,Kepler-692 b,Kepler-692,294.776807,40.153703,21.812935,3.11,9.85,5440.0,0.860,0.900,991.336,2016,Planet,0,NASA Exoplanet Archive
4,121598758,Kepler-150 c,Kepler-150,288.234103,40.520897,7.381998,3.69,13.20,5560.0,0.939,0.956,891.092,2014,Planet,0,NASA Exoplanet Archive


In [6]:
def output_file(tic_id, sector):

    return LIGHTCURVE_DIR / f"TIC_{tic_id}_Sector_{sector}.parquet"


def already_downloaded(tic_id, sector):

    return output_file(tic_id, sector).exists()


def save_lightcurve(df, tic_id, sector):

    df.to_parquet(
        output_file(tic_id, sector),
        index=False
    )

In [7]:
def search_lightcurve(tic_id):

    target = f"TIC {tic_id}"

    result = lk.search_lightcurve(
        target,
        mission=MISSION,
        author=AUTHOR
    )

    return result

In [8]:
def download_lightcurve(search_result):

    if len(search_result) == 0:
        return None

    lc = search_result.download()

    return lc

In [9]:
def lc_to_dataframe(lc, tic_id):

    # Extract metadata safely
    sector = lc.meta.get("SECTOR", np.nan)
    author = lc.meta.get("AUTHOR", "Unknown")
    mission = lc.meta.get("MISSION", "TESS")
    target_name = lc.meta.get("TARGETID", f"TIC {tic_id}")
    cadence = lc.meta.get("EXPOSURE", np.nan)

    def fix_byteorder(arr):
        if hasattr(arr, 'dtype') and arr.dtype.byteorder not in ('=', '|'):
            return arr.astype(arr.dtype.newbyteorder('='))
        return arr

    time = fix_byteorder(lc.time.value)
    flux = fix_byteorder(lc.flux.value)
    flux_err = fix_byteorder(lc.flux_err.value)
    quality = fix_byteorder(lc.quality)

    df = pd.DataFrame({

        "tic_id": tic_id,

        "sector": sector,

        "mission": mission,

        "author": author,

        "target_name": target_name,

        "cadence_sec": cadence,

        "time": time,

        "flux": flux,

        "flux_err": flux_err,

        "quality": quality

    })

    return df

In [10]:
download_log = []

for _, row in tqdm(catalog.iterrows(), total=len(catalog)):

    tic_id = row["tic_id"]

    try:

        search = search_lightcurve(tic_id)

        if len(search) == 0:

            download_log.append({

                "tic_id": tic_id,

                "status": "Not Found"

            })

            continue

        lc = download_lightcurve(search)

        sector = lc.meta.get("SECTOR", "Unknown")

        if already_downloaded(tic_id, sector):

            download_log.append({

                "tic_id": tic_id,

                "status": "Skipped"

            })

            continue

        df = lc_to_dataframe(lc, tic_id)

        save_lightcurve(df, tic_id, sector)

        download_log.append({

            "tic_id": tic_id,

            "status": "Success",

            "sector": sector,

            "points": len(df)

        })

        del lc
        del df

        gc.collect()

    except Exception as e:

        logger.error(f"{tic_id} : {e}")

        download_log.append({

            "tic_id": tic_id,

            "status": "Failed",

            "error": str(e)

        })

  0%|          | 0/100 [00:00<?, ?it/s]

0% (22/18810) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-07-23 10:56:33,002 | INFO | 0% (22/18810) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (41/18597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-07-23 10:56:34,216 | INFO | 0% (41/18597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (41/18597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-07-23 10:56:35,369 | INFO | 0% (41/18597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (41/18597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-07-23 10:56:44,713 | INFO | 0% (41/18597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (22/18810) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-07-23 10

In [11]:
report = pd.DataFrame(download_log)

report.to_csv(
    ROOT_DIR / "reports" / "download_report.csv",
    index=False
)

report.head()

,tic_id,status,sector,points,error
0,158722002,Success,74.0,18788.0,NaN
1,121988890,Success,80.0,18556.0,NaN
2,350813882,Success,80.0,18556.0,NaN
3,138728891,Not Found,NaN,NaN,NaN
4,121598758,Success,80.0,18556.0,NaN


In [12]:
import lightkurve as lk

tic = catalog.iloc[0]["tic_id"]

print("TIC:", tic)

search = lk.search_lightcurve(
    f"TIC {tic}",
    mission="TESS"
)

print(search)

TIC: 158722002
SearchResult containing 16 data products.

 #     mission     year   author  exptime target_name distance
                                     s                 arcsec 
--- -------------- ---- --------- ------- ----------- --------
  0 TESS Sector 74 2024      SPOC     120   158722002      0.0
  1 TESS Sector 75 2024      SPOC     120   158722002      0.0
  2 TESS Sector 80 2024      SPOC     120   158722002      0.0
  3 TESS Sector 81 2024      SPOC     120   158722002      0.0
  4 TESS Sector 82 2024      SPOC     120   158722002      0.0
  5 TESS Sector 74 2024 TESS-SPOC     200   158722002      0.0
  6 TESS Sector 75 2024 TESS-SPOC     200   158722002      0.0
  7 TESS Sector 80 2024 TESS-SPOC     200   158722002      0.0
  8 TESS Sector 81 2024 TESS-SPOC     200   158722002      0.0
  9 TESS Sector 82 2024 TESS-SPOC     200   158722002      0.0
 10 TESS Sector 40 2021     CDIPS    1800   158722002      0.0
 11 TESS Sector 41 2021     CDIPS    1800   158722002      0

In [13]:
print(search.table)

intentType obs_collection provenance_name ...  #  year sort_order
                                          ...                    
---------- -------------- --------------- ... --- ---- ----------
   science           TESS            SPOC ...   0 2024          1
   science           TESS            SPOC ...   1 2024          1
   science           TESS            SPOC ...   2 2024          1
   science           TESS            SPOC ...   3 2024          1
   science           TESS            SPOC ...   4 2024          1
   science           HLSP       TESS-SPOC ...   5 2024          2
   science           HLSP       TESS-SPOC ...   6 2024          2
   science           HLSP       TESS-SPOC ...   7 2024          2
   science           HLSP       TESS-SPOC ...   8 2024          2
   science           HLSP       TESS-SPOC ...   9 2024          2
   science           HLSP           CDIPS ...  10 2021          9
   science           HLSP           CDIPS ...  11 2021          9
   science

In [14]:
lc = search.download()

print(lc)

0% (22/18810) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-07-23 11:20:54,000 | INFO | 0% (22/18810) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


       time             flux      ...   pos_corr1      pos_corr2   
                    electron / s  ...      pix            pix      
------------------ -------------- ... -------------- --------------
3312.8619411872546  4.8938211e+02 ... -1.0616370e-02  2.9418696e-04
3312.8633300545566  4.7517654e+02 ... -4.4883646e-02  1.2689924e-02
3312.8647189216244  4.8428027e+02 ... -2.3553183e-02 -2.5926451e-03
 3312.866107788926  4.7133551e+02 ...  9.6208649e-03 -1.2643359e-02
3312.8674966559947  4.9106375e+02 ... -6.2997796e-02  1.4230090e-02
3312.8688855232954  4.8189792e+02 ... -6.7708403e-04 -6.8615363e-03
 3312.870274390597  4.8661029e+02 ... -4.0195961e-02  8.2047498e-03
3312.8716632576657  4.8705084e+02 ...  2.0812165e-03 -1.7876185e-02
3312.8730521249668  4.7663110e+02 ... -6.9689706e-02  2.2467177e-02
               ...            ... ...            ...            ...
 3339.553472561022  4.8908813e+02 ... -3.1746078e-02 -1.2885166e-02
3339.5548614590566  4.8515509e+02 ... -3.6329307

In [15]:
search = lk.search_lightcurve(f"TIC {catalog.iloc[0]['tic_id']}", mission="TESS")
print(search)
print(search.table)

SearchResult containing 16 data products.

 #     mission     year   author  exptime target_name distance
                                     s                 arcsec 
--- -------------- ---- --------- ------- ----------- --------
  0 TESS Sector 74 2024      SPOC     120   158722002      0.0
  1 TESS Sector 75 2024      SPOC     120   158722002      0.0
  2 TESS Sector 80 2024      SPOC     120   158722002      0.0
  3 TESS Sector 81 2024      SPOC     120   158722002      0.0
  4 TESS Sector 82 2024      SPOC     120   158722002      0.0
  5 TESS Sector 74 2024 TESS-SPOC     200   158722002      0.0
  6 TESS Sector 75 2024 TESS-SPOC     200   158722002      0.0
  7 TESS Sector 80 2024 TESS-SPOC     200   158722002      0.0
  8 TESS Sector 81 2024 TESS-SPOC     200   158722002      0.0
  9 TESS Sector 82 2024 TESS-SPOC     200   158722002      0.0
 10 TESS Sector 40 2021     CDIPS    1800   158722002      0.0
 11 TESS Sector 41 2021     CDIPS    1800   158722002      0.0
 12 TESS Sec

Fixes something that I might be worried about

In [16]:
def download_lightcurve(search_result):

    if len(search_result) == 0:
        return None

    # Prefer official SPOC products
    priority = ["SPOC", "TESS-SPOC", "QLP", "CDIPS", "TASOC"]

    for author in priority:
        matches = search_result[search_result.author == author]
        if len(matches) > 0:
            try:
                return matches[0].download()
            except Exception:
                continue

    return None